In [1]:
%pip install -r requirements.txt

Usage: __main__.py [options]

ERROR: Invalid requirement: python3.12 -m venv .venv
__main__.py: error: no such option: -m

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os 
import secrets

#Utilizing xor to help with encryption and decryption 

def xor_bytes(data: bytes, key: bytes) -> bytes:  
    if len(data) != len(key): 
        raise ValueError("data and key must have the same length")
    return bytes(d ^ k for d, k in zip(data, key)) #Creates ciphertext based on key
 
def alice(cipher_file="ciphertext.hex", key_file="key.hex"):
    plaintext = input("Enter your message Alice: ").encode() #Inputting for alice message
    key = secrets.token_bytes(len(plaintext)) #plaintext gets encrypted with a random key
    ciphertext = xor_bytes(plaintext, key) #Creates ciphertext
    print("Cipher text (hex):", ciphertext.hex()) 
    with open(cipher_file, "w") as f:  #Saves ciphertext to file
        f.write(ciphertext.hex())

    with open(key_file, "w") as f:  #Saves key to file 
        f.write(key.hex())
    
    print("Saved ciphertext to file: '{cipher_file}' and key to '{key_file}'")


In [5]:
def bob(cipher_file="ciphertext.hex", key_file = "key.hex"):
    with open(key_file) as f: #Reads key from key file 
        key = bytes.fromhex(f.read().strip())
    with open(cipher_file) as f: #Reads cipher from ciphertext file 
        ciphertext= bytes.fromhex(f.read().strip())
    plaintext = xor_bytes(ciphertext, key).decode()
    print("Bob decrypted:", plaintext)
    return plaintext

In [6]:
#Print statement 

alice() # Enter a message 
bob() #Will read the key and display the plaintext
print("Files:", [f for f in ("ciphertext.hex", "key.hex") if os.path.exists(f)]) #Checks for file initially then list cipher and keys
print("key.hex       :", open("key.hex").read())
print("ciphertext.hex:", open("ciphertext.hex").read())


Cipher text (hex): f8e6f91513fbe4d1ad16d7f14b48dcaa745dd9196c7b473beaea7138ec1d758e3176cf0011bb2ef4bed006a8234faebbc622fa1b3878a0330cbf47
Saved ciphertext to file: '{cipher_file}' and key to '{key_file}'


FileNotFoundError: [Errno 2] No such file or directory: 'key.hex'

In [ ]:
#Problem 3: Implementing Many-Time Pad 

In [ ]:
MESSAGES = [
    "The one-time pad is perfectly secure only when the key is random, secret, as long as the message and never reused.",
    "During the Second World War Soviet intelligence services reused some of their pages, and analysts later exploited that mistake.",
    "Meet me tomorrow at the old library near the river, and please bring along the documents we discussed last week.",
    "A stream cipher XORs a keystream with the plaintext, so reusing the same keystream leaks the XOR of two messages.",
    "Never trust a system whose security depends on the secrecy of its design rather than on the secrecy of its key.",
    "The attacker does not need the key at all, because XORing two ciphertexts cancels the key and exposes both messages.",
    "Cryptography is hard to get right, and the smallest implementation mistake can quietly destroy an otherwise perfect scheme.",
    "Please confirm that the package arrived safely at the harbor and that nobody followed you on the way to the station.",
    "Spaces are the weakness here, since a space XORed with a letter simply flips the case of that letter in the result.",
    "Always generate fresh random keys from a cryptographically secure source, and destroy every key right after using it.",
]

def many_time_pad(messages, key=None):
    """Encrypt every message with the SAME key (insecure on purpose)."""
    plaintexts = [m.encode("utf-8") for m in messages]
    if key is None:
        key = secrets.token_bytes(max(len(p) for p in plaintexts))   # long enough for all
    ciphertexts = [xor_bytes(p, key[:len(p)]) for p in plaintexts]   # same key, every time
    return plaintexts, key, ciphertexts


def save_many_time_pad(plaintexts, key, ciphertexts, filename="many_time_pad.txt"):
    """Write plaintexts, key and ciphertexts (all hex) into one file."""
    with open(filename, "w") as f:
        f.write(f"KEY: {key.hex()}\n\n")
        for i, (p, c) in enumerate(zip(plaintexts, ciphertexts), start=1):
            f.write(f"PLAINTEXT_{i}: {p.hex()}\n")
            f.write(f"CIPHERTEXT_{i}: {c.hex()}\n\n")


plaintexts, key, ciphertexts = many_time_pad(MESSAGES)
save_many_time_pad(plaintexts, key, ciphertexts)

print(f"Key length: {len(key)} bytes; message lengths: {[len(p) for p in plaintexts]}")
print("First ciphertext (hex):", ciphertexts[0].hex()[:64], "...")